# Session 18: Finding structure without labels

Segment the Lakeside Mart loyalty customers with recency, frequency and spend. Walk-in orders have no customer id and are left out. Pandas cells check exact counts. The k-means and elbow cells check tolerant ranges, because scikit-learn groups will be close to the class numbers and differ a little.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Keep loyalty orders
Read `orders.csv` into `orders`, turn `order_date` into dates, and keep only the orders that have a `customer_id`. Call the result `loyal`. Walk-in orders have a blank id and are left out.

In [ ]:
import pandas as pd
import numpy as np

orders = pd.read_csv('orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])
loyal = orders.dropna(subset=['customer_id'])
print(len(loyal), loyal['customer_id'].nunique())

In [ ]:
assert len(orders) - len(loyal) == 9171, 'Walk-in orders (blank customer_id) must be dropped'
assert len(loyal) == 21058, 'loyal should have 21058 orders. Drop the rows where customer_id is blank'
assert loyal['customer_id'].nunique() == 2499, 'There should be 2499 loyalty customers'
print('Task 1 correct')

## Task 2. Build recency, frequency and spend
Make `rfm`, one row per customer id, with columns `recency` (days from the customer's last order to the last order date in the file), `frequency` (number of orders) and `spend` (sum of `order_value_ugx`), in that order.

In [ ]:
end = loyal['order_date'].max()
rfm = loyal.groupby('customer_id').agg(
    last_order=('order_date', 'max'),
    frequency=('order_id', 'count'),
    spend=('order_value_ugx', 'sum'))
rfm['recency'] = (end - rfm['last_order']).dt.days
rfm = rfm[['recency', 'frequency', 'spend']]
rfm.head()

In [ ]:
assert rfm.shape == (2499, 3), 'One row per loyalty customer, three columns'
assert list(rfm.columns) == ['recency', 'frequency', 'spend'], 'The columns must be recency, frequency, spend in that order'
assert rfm.loc['C0001', 'recency'] == 6 and rfm.loc['C0001', 'frequency'] == 15, 'Check customer C0001: recency 6 days, 15 orders'
assert rfm['frequency'].sum() == 21058, 'frequency is the count of orders for each customer'
assert rfm['spend'].sum() == 998818340, 'spend is the sum of order_value_ugx for each customer'
print('Task 2 correct')

## Task 3. Log and scale
Make `logged` with `np.log1p(rfm)`. Then make `scaled` by subtracting each column's mean and dividing by its standard deviation. Every column of `scaled` should have mean 0 and spread 1.

In [ ]:
logged = np.log1p(rfm)
scaled = (logged - logged.mean()) / logged.std()
print(scaled.mean().round(3))
print(scaled.std().round(3))

In [ ]:
assert abs(scaled.mean()).max() < 1e-6, 'Subtract the column mean'
assert abs(scaled.std() - 1).max() < 1e-6, 'Divide by the column standard deviation'
assert abs(np.log1p(rfm['spend']).skew()) < 0.1, 'logged should come from np.log1p(rfm)'
print('Task 3 correct')

## Task 4. Cluster with k-means
Scale `logged` with scikit-learn's `StandardScaler` into `X`, then use `KMeans` with 4 clusters, `n_init=10` and `random_state=42`. Store each customer's group in `rfm['segment']`. Your groups will be close to the ones in class and will differ a little, and the numbers 0 to 3 are only labels.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = StandardScaler().fit_transform(logged)
km = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm['segment'] = km.fit_predict(X)
print(rfm['segment'].value_counts())

In [ ]:
# sklearn result: tolerant check, because exact groups can differ
sizes = rfm['segment'].value_counts()
assert len(sizes) == 4, 'Four segments expected: use n_clusters=4'
assert sizes.sum() == 2499, 'Every customer needs a segment'
assert sizes.min() > 250 and sizes.max() < 1300, 'Sizes should be near 460 to 880 in a typical run. Cluster the scaled columns in X'
print('Task 4 correct')

## Task 5. Choose k with the elbow
For k from 1 to 8, fit `KMeans` on `X` (`n_init=10`, `random_state=42`) and store each fit's `inertia_`, the spread inside the groups, in the list `inertias`. Plot it against k. Where does the line bend, and would you still choose 4?

In [ ]:
# uses KMeans from scikit-learn (sklearn) above
import matplotlib.pyplot as plt

inertias = []
for k in range(1, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(m.inertia_)
plt.plot(range(1, 9), inertias, marker='o')
plt.xlabel('Number of clusters k')
plt.ylabel('Spread inside the groups')
plt.show()

In [ ]:
# sklearn result: tolerant check, because exact groups can differ
assert len(inertias) == 8, 'Add one value for each k from 1 to 8'
assert all(a > b for a, b in zip(inertias, inertias[1:])), 'More groups always shrink the spread. Check you fitted a new KMeans with k clusters each time'
assert 7000 < inertias[0] < 8000, 'With k = 1 the spread is about 3 columns times 2499 customers. Fit on X'
print('Task 5 correct')

## Task 6. Profile each segment
Make `profile`, a table with one row per segment and four columns: `customers` (the number of customers), `recency`, `orders` and `spend` (the averages of recency, frequency and spend). Round to whole numbers.

In [ ]:
# uses the segment column from scikit-learn (sklearn) above
profile = rfm.groupby('segment').agg(
    customers=('spend', 'size'),
    recency=('recency', 'mean'),
    orders=('frequency', 'mean'),
    spend=('spend', 'mean')).round()
profile

In [ ]:
# sklearn result: tolerant check, because exact groups can differ
assert profile['customers'].sum() == 2499, 'Customer counts must add up to all 2499 loyalty customers'
assert profile['spend'].max() > 2 * profile['spend'].min(), 'spend should be the average spend. The best segment should spend over twice the weakest'
assert profile['recency'].max() > 100 and profile['recency'].min() < 60, 'recency should be the average days since the last order'
print('Task 6 correct')

## Task 7. Flag unusual orders
Using all of `orders` (walk-ins included), store the 99th percentile of `order_value_ugx` in `p99` and keep the orders above it in `big`.

In [ ]:
p99 = orders['order_value_ugx'].quantile(0.99)
big = orders[orders['order_value_ugx'] > p99]
print(round(p99), len(big))

In [ ]:
assert round(p99) == 468620, 'p99 is the 99th percentile (0.99 quantile) of order_value_ugx over all orders'
assert len(big) == 303, 'Keep orders strictly above the percentile: 303 orders'
print('Task 7 correct')

## Task 8. Name and describe each segment
Double click this cell and write one line per segment, using business words in place of group numbers. Say what the customers do, how many there are, and what you would do for them. A table like the one below is fine.

| Name | Who they are | What I would do |
|---|---|---|
| (name) | (recency, orders, spend, count) | (one action) |

Then write one sentence: you flagged 303 big orders. What would you check before treating any of them as a mistake?